# 03 — Embedding Space EDA

Explore the ChromaDB vector store after ingestion: inspect stored chunks, visualise the embedding space (UMAP), and test semantic search.

In [ ]:
from pathlib import Path

import chromadb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from umap import UMAP

sns.set_style("darkgrid")

In [ ]:
DATA_DIR = Path('../data')
collection_name = "actuarial_abstracts"

## 1. Connect to ChromaDB

In [ ]:
client = chromadb.PersistentClient(path=DATA_DIR / "chromadb")
collection = client.get_collection(collection_name)

print(f"Collection: {collection.name}")
print(f"Total vectors: {collection.count()}")

In [ ]:
sample = collection.get(limit=1, include=["metadatas", "documents"])

print("Metadata fields:", list(sample["metadatas"][0].keys()))
print("\nExample values:")
for k, v in sample["metadatas"][0].items():
    print(f"  {k}: {v}")

## 2. Inspect stored chunks

In [ ]:
# Peek at the first few entries
peek = collection.peek(limit=5)

for i, (doc, meta) in enumerate(zip(peek["documents"], peek["metadatas"])):
    print(f"--- Chunk {i} ---")
    print(f"  Title: {meta.get('title', '?')}")
    print(f"  Year: {meta.get('year', '?')} | Company: {meta.get('company', '?')}")
    print(f"  Text: {doc[:150]}...\n")

## 3. Load all embeddings

In [ ]:
# Retrieve all data (embeddings + metadata)
all_data = collection.get(include=["embeddings", "metadatas"])

embeddings = np.array(all_data["embeddings"])
metadatas = all_data["metadatas"]

print(f"Embeddings shape: {embeddings.shape}")
print(f"Dimension: {embeddings.shape[1]}")

## 4. UMAP — 2D projection

Reduce 768 dimensions → 2D to visualise cluster structure.

In [ ]:
reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_2d = reducer.fit_transform(embeddings)

df_emb = pd.DataFrame(X_2d, columns=["x", "y"])
df_emb["company"] = [m.get("company", "Unknown") for m in metadatas]
df_emb["year"] = [m.get("year", "Unknown") for m in metadatas]
df_emb["title"] = [m.get("title", "")[:50] for m in metadatas]

In [ ]:
plt.figure(figsize=(14, 9))
plt.scatter(df_emb["x"], df_emb["y"], s=8, alpha=0.6, c="steelblue")
plt.title("Embedding space — UMAP 2D projection")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## 5. Color by company (top 10)

In [ ]:
top_companies = df_emb["company"].value_counts().head(10).index.tolist()
df_top = df_emb[df_emb["company"].isin(top_companies)]

plt.figure(figsize=(14, 9))
for company in top_companies:
    subset = df_top[df_top["company"] == company]
    plt.scatter(subset["x"], subset["y"], s=12, alpha=0.6, label=company)

plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.title("Embedding space — colored by company (top 10)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## 6. Semantic search

Test retrieval quality: embed a query and find the closest chunks.

In [ ]:
from actuarial_genai_rag.ingestion.config import load_config
from actuarial_genai_rag.retrieval.embedder import Embedder

config = load_config("../config/ingestion.yaml")
embedder = Embedder(config.embedding)

In [ ]:
query = "modélisation du risque de longévité en assurance vie"
query_emb = embedder.embed_query(query)

results = collection.query(
    query_embeddings=[query_emb.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: '{query}'\n")
for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"[distance={dist:.4f}] {meta.get('title', '?')} ({meta.get('year', '?')})")
    print(f"  {doc[:150]}...\n")

## 7. Filter by metadata

ChromaDB supports `where` clauses to filter results.

In [ ]:
# Search only among recent theses (year >= 2020)
results_filtered = collection.query(
    query_embeddings=[query_emb.tolist()],
    n_results=5,
    where={"year": 2020},
    include=["documents", "metadatas", "distances"],
)

print("Filtered results (year >= 2020):\n")
for doc, meta, dist in zip(
    results_filtered["documents"][0],
    results_filtered["metadatas"][0],
    results_filtered["distances"][0],
):
    print(f"[{dist:.4f}] {meta.get('title', '?')} ({meta.get('year', '?')})")